# 01 · ABSA: aspect extraction (DistilBERT, BIO) + aspect sentiment (ALBERT)
Buổi 12. Trains on **SemEval-2014** (laptop + restaurant, English) and **UIT-ViSD4SA** (Vietnamese), pooled and re-split 70/15/15.

* ATE: `distilbert-base-multilingual-cased`, token classification with BIO tags (`B-ASP`/`I-ASP` for English, category-typed tags such as `B-BATTERY` for Vietnamese).
* ATSC: `cservan/multilingual-albert-base-cased-32k`, sentence-pair classification (sentence, aspect) → negative / neutral / positive, class-weighted loss.
* Target: F1 > 0.80. Results are pushed to the Hub and merged into `metrics.json`, which the web app's Models page reads.

About 35 minutes on a T4.

In [ ]:
#@title Setup — run once per session (Runtime > Change runtime type > T4 GPU)
REPO_URL = "https://github.com/YoungsterDumb/reviewlens"  # @param {type:"string"}
HF_NAMESPACE = "DatDuc"  # @param {type:"string"}  your Hugging Face username
import os, sys, subprocess
if not os.path.exists("/content/reviewlens"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "/content/reviewlens"], check=True)
else:
    subprocess.run(["git", "-C", "/content/reviewlens", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/reviewlens/training/requirements.txt"], check=True)
sys.path.insert(0, "/content/reviewlens/training")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch the runtime to T4 GPU")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a token with WRITE access (huggingface.co/settings/tokens)
METRICS_REPO = f"{HF_NAMESPACE}/reviewlens-metrics"

## 1. Data: load, de-duplicate, split 70/15/15

In [ ]:
from pathlib import Path
from collections import Counter
from reviewlens_train.data import load_absa, to_bio
splits = load_absa(Path("/content/data"))
for name, rows in splits.items():
    print(f"{name:10s}", len(rows), dict(Counter(r["lang"] for r in rows)))
example = to_bio([r for r in splits["train"] if r["lang"] == "vi"][:1])[0]
print(list(zip(example["tokens"], example["tags"]))[:20])

## 2. Aspect term extraction (ATE)

In [ ]:
from reviewlens_train.absa_train import train_ate
ate_trainer, ate_tok, ate_metrics = train_ate(splits, Path("/content/out/ate"), epochs=5)
ate_metrics

In [ ]:
from reviewlens_train.hub import push_model
push_model(ate_trainer.model, ate_tok, f"{HF_NAMESPACE}/reviewlens-ate", "ATE: DistilBERT multilingual, BIO")

## 3. Aspect term sentiment classification (ATSC)

In [ ]:
from reviewlens_train.absa_train import train_atsc
atsc_trainer, atsc_tok, atsc_metrics, pairs = train_atsc(splits, Path("/content/out/atsc"), epochs=4)
atsc_metrics

In [ ]:
push_model(atsc_trainer.model, atsc_tok, f"{HF_NAMESPACE}/reviewlens-atsc", "ATSC: multilingual ALBERT, sentence pairs")

## 4. Publish metrics and try the full ATE → ATSC pipeline

In [ ]:
from reviewlens_train.hub import update_metrics
update_metrics(METRICS_REPO, {**ate_metrics, **atsc_metrics})

In [ ]:
from app.models.text_models import AspectExtractor, SentimentClassifier
from app.pipeline.fusion import run_absa
ate = AspectExtractor(f"{HF_NAMESPACE}/reviewlens-ate")
atsc = SentimentClassifier(f"{HF_NAMESPACE}/reviewlens-atsc")
for review, lang in [("The screen is gorgeous but the battery barely lasts four hours.", "en"),
                     ("Pin trâu, màn hình đẹp nhưng camera chụp đêm bị nhoè.", "vi")]:
    aspects, overall = run_absa(ate, atsc, review, lang)
    print(review, "->", [(a["term"], a["category"], a["sentiment"], a["confidence"]) for a in aspects], overall["label"])